# 08 - معالجة ملفات الجداول CSV و Excel (CSV & Excel Tabular Data Ingestion)

مرحباً بك في الدرس الثامن من مسار **Data Ingestion** ضمن كورس **RAG Mastery**!

في هذا الدرس، ننتقل للتعامل مع **البيانات المهيكلة والجدولية (Structured Tabular Data)**:
1. **`CSVLoader`**: الأداة القياسية في LangChain لتحميل صفوف الـ CSV مباشرة ككائنات `Document`.
2. **المعالجة الذكية المخصصة عبر `Pandas`**: تحويل الصفوف إلى جمل سردية دلالية وإثراء الـ Metadata لتسهيل التصفية البرمجية (`Metadata Filtering`).
3. **معالجة دفاتر Excel متعددة أوراق العمل (`Multi-Sheet Excel`)**: استخراج الجداول من عدة Sheets مع حفظ بيانات المصدر.

## 1. استيراد المكتبات والتحقق من البيئة

In [ ]:
import os
from pathlib import Path
import pandas as pd
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.document_loaders import CSVLoader

print(f"Pandas version: {pd.__version__}")
print("LangChain Document Loaders imported successfully!")

## 2. إعداد وفحص بيانات الاختبار (CSV & Multi-Sheet Excel)

سنقوم بفحص ملف كتالوج المنتجات `data/products.csv` وملف المخزون متعدد الأوراق `data/inventory.xlsx`.

In [ ]:
csv_file = Path("data/products.csv")
excel_file = Path("data/inventory.xlsx")

print(f"[+] CSV File exists: {csv_file.exists()} (Size: {csv_file.stat().st_size if csv_file.exists() else 0} bytes)")
print(f"[+] Excel File exists: {excel_file.exists()} (Size: {excel_file.stat().st_size if excel_file.exists() else 0} bytes)")

# استعراض سريع لبيانات الـ CSV باستخدام Pandas
df_preview = pd.read_csv(csv_file)
display(df_preview.head())

## 3. الطريقة الأولى: القراءة التلقائية للصفوف عبر `CSVLoader` (Row-Based Loading)

يقوم `CSVLoader` بتحميل كل صف كـ `Document` منفصل، ويضع رقم الصف في الـ Metadata.

In [ ]:
try:
    loader = CSVLoader(
        file_path=str(csv_file),
        encoding="utf-8",
        csv_args={
            "delimiter": ",",
            "quotechar": '"'
        }
    )
    csv_docs = loader.load()

    print(f"[+] Successfully loaded {len(csv_docs)} rows as Documents.\n")
    
    for i, doc in enumerate(csv_docs[:3], 1):
        print(f"--- Document #{i} [Row: {doc.metadata.get('row')}] ---")
        print(f"Content:\n{doc.page_content}")
        print(f"Metadata: {doc.metadata}\n")
except Exception as e:
    print(f"[-] Error loading CSV: {e}")

## 4. الطريقة الثانية: المعالجة الذكية والمخصصة عبر `Pandas` (Semantic Row Serialization)

في بيئات الإنتاج، يُفضَّل تحويل صفوف الجدول إلى سياق سردي دلالي غني (`Natural Language Context`) بدلاً من مجرد مفاتيح وقيم جافة، مع استخراج الأعمدة الرقمية والتصنيفية داخل الـ `metadata` لتمكين الفلترة الدقيقة.

In [ ]:
def process_csv_intelligently(file_path: str) -> List[Document]:
    df = pd.read_csv(file_path)
    documents: List[Document] = []

    for idx, row in df.iterrows():
        p_name = row["product"]
        cat = row["category"]
        price = row["price"]
        desc = row["description"]

        # صياغة نص سردي متكامل يفهمه نموذج التضمين بدقة عالية
        content = (
            f"Product Name: {p_name}\n"
            f"Category: {cat}\n"
            f"Retail Price: ${price}\n"
            f"Detailed Description: {desc}"
        )

        # إثراء الميتا-داتا للاستفادة منها في الفلترة بقواعد البيانات المتجهة
        metadata = {
            "source": Path(file_path).name,
            "row_index": int(idx),
            "product_name": str(p_name),
            "category": str(cat),
            "price": float(price),
            "is_premium": float(price) >= 1000.0
        }

        documents.append(Document(page_content=content, metadata=metadata))

    return documents

intelligent_docs = process_csv_intelligently(str(csv_file))
print(f"[+] Generated {len(intelligent_docs)} intelligent documents.\n")

# استعراض أول مستند مع الميتا-داتا الغنية
print("--- Sample Enriched Document ---")
print(f"Content:\n{intelligent_docs[0].page_content}")
print(f"Enriched Metadata:\n{intelligent_docs[0].metadata}")

## 5. الطريقة الثالثة: معالجة دفاتر Excel متعددة الأوراق (`Multi-Sheet Excel`)

ملفات Excel تحتوي غالباً على أوراق عمل متعددة (`Sheets`). نستخدم `pd.ExcelFile` لقراءة كل ورقة عمل وربط اسمها بالمستندات المستخرجة.

In [ ]:
def process_multisheet_excel(excel_path: str) -> List[Document]:
    excel_app = pd.ExcelFile(excel_path)
    sheet_names = excel_app.sheet_names
    print(f"[+] Found {len(sheet_names)} sheets in Excel workbook: {sheet_names}")

    excel_docs: List[Document] = []

    for sheet in sheet_names:
        sheet_df = pd.read_excel(excel_app, sheet_name=sheet)
        
        for idx, row in sheet_df.iterrows():
            # تحويل خلايا الصف إلى تمثيل مهيكل
            row_repr = " | ".join([f"{col}: {val}" for col, val in row.items()])
            
            meta = {
                "source": Path(excel_path).name,
                "sheet_name": sheet,
                "row_index": int(idx),
                "columns_count": len(sheet_df.columns)
            }
            excel_docs.append(Document(page_content=row_repr, metadata=meta))

    return excel_docs

excel_documents = process_multisheet_excel(str(excel_file))
print(f"\n[+] Total rows extracted across all sheets: {len(excel_documents)}\n")

for doc in excel_documents[:4]:
    print(f"[Sheet: {doc.metadata['sheet_name']} | Row {doc.metadata['row_index']}] -> {doc.page_content}")

## 6. الخلاصة والمقارنة المعمارية

| الاستراتيجية | الأنسب لـ | المزايا | القيود |
| :--- | :--- | :--- | :--- |
| **`CSVLoader`** | جداول بسيطة ومستقلة الصفوف | سريع ومباشر | سياق جاف يفتقر للدلالات اللغوية |
| **`Pandas Custom Processing`** | كتالوجات المنتجات وقواعد المعرفة | سياق لغوي ممتاز + فلترة دقيقة بالميتا-داتا | يتطلب كتابة كود مخصص لكل نوع جدول |
| **`Multi-Sheet Excel`** | التقارير المالية والإدارية المعقدة | يغطي جميع أوراق العمل ويحفظ هيكلها | قد يحتاج معالجة إضافية للجداول المدمجة |